# Programming Assignment: Production-Ready Imputation Pipeline

Welcome to the **Production-Ready Imputation Pipeline** assignment!

In production, imputation cannot be a one-off step — it must be a first-class component of the preprocessing pipeline. This means:

- **Encapsulation**: Imputation logic is wrapped in a class with a clean `fit/transform` API so it composes naturally with sklearn
- **Leakage prevention**: Statistics learned in `fit()` are frozen; `transform()` only applies those stored values
- **Edge case robustness**: All-missing columns and constant columns must be detected and handled gracefully before modelling
- **Auditability**: Every imputation decision should be logged (column name, strategy, fill value, cells imputed)
- **Serialisability**: The fitted pipeline can be saved and re-loaded with `joblib` for serving

**What You Will Do in This Assignment:**

* Build a `ConfigurableImputer` class from scratch with `fit/transform/fit_transform`
* Write an edge-case detector for pathological columns
* Implement an sklearn-compatible `SklearnImputer` (inheriting `BaseEstimator`, `TransformerMixin`)
* Implement a `ProductionImputer` that enforces strict data-leakage prevention
* Build an `ImputationLogger` that records every imputation decision
* Assemble a complete preprocessing pipeline with `ColumnTransformer` and `Pipeline`

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* **Avoid using global variables** unless absolutely necessary. The grader tests your code in an isolated environment.

---

## Table of Contents
- [Imports](#imports)
- [1 - Configurable Imputer Class](#1---configurable-imputer-class)
    - **[Exercise 1 - `ConfigurableImputer`](#exercise-1---configurableimputer)**
- [2 - Handling Edge Cases](#2---handling-edge-cases)
    - **[Exercise 2 - `handle_edge_cases`](#exercise-2---handle_edge_cases)**
- [3 - Sklearn Pipeline Integration](#3---sklearn-pipeline-integration)
    - **[Exercise 3 - `SklearnImputer`](#exercise-3---sklearnimputer)**
- [4 - Fit/Transform Pattern](#4---fittransform-pattern)
    - **[Exercise 4 - `ProductionImputer`](#exercise-4---productionimputer)**
- [5 - Imputation Logging](#5---imputation-logging)
    - **[Exercise 5 - `ImputationLogger`](#exercise-5---imputationlogger)**
- [6 - Complete Preprocessing Pipeline](#6---complete-preprocessing-pipeline)
    - **[Exercise 6 - `create_preprocessing_pipeline`](#exercise-6---create_preprocessing_pipeline)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
import joblib

In [ ]:
import helper_utils
import unittests

## Data Preparation

We use `helper_utils.generate_train_test_data()` to produce a mixed-type dataset (numeric + categorical) with missing values, representative of a real-world ingestion scenario.

In [ ]:
X_train, X_test, y_train, y_test = helper_utils.generate_train_test_data(random_state=42)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"\nDtypes:\n{X_train.dtypes}")
print(f"\nMissing (train):\n{X_train.isnull().sum()}")

In [ ]:
helper_utils.visualize_missing_patterns(X_train)

<a name='1---configurable-imputer-class'></a>
## 1 - Configurable Imputer Class

We start by building a **pure Python** imputer without relying on sklearn. This reinforces the mechanics of the fit/transform pattern: `fit()` reads the training data and stores statistics; `transform()` applies those stored values to any dataset without touching the training data again.

<a name='exercise-1---configurableimputer'></a>
### **Exercise 1 - `ConfigurableImputer`**

**Your Task:**

Implement a `ConfigurableImputer` class with the following API:
- `__init__(strategy='mean', fill_value=None)` — store the strategy
- `fit(X)` — compute and store imputation statistics in `self.statistics_` (dict mapping column name → fill value); return `self`
- `transform(X)` — return a copy of X with missing values filled using `self.statistics_`; raise `ValueError` if `fit()` has not been called
- `fit_transform(X)` — convenience wrapper: `self.fit(X).transform(X)`

Supported strategies: `'mean'`, `'median'`, `'mode'`, `'constant'`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
def fit(self, X):
    df = pd.DataFrame(X)
    self.statistics_ = {}
    for col in df.columns:
        if self.strategy == 'mean':
            self.statistics_[col] = df[col].mean()
        elif self.strategy == 'median':
            self.statistics_[col] = df[col].median()
        elif self.strategy == 'mode':
            self.statistics_[col] = df[col].mode()[0]
        else:  # constant
            self.statistics_[col] = self.fill_value
    return self

def transform(self, X):
    if not hasattr(self, 'statistics_'):
        raise ValueError('Imputer has not been fitted yet. Call fit() first.')
    result = pd.DataFrame(X).copy()
    for col, value in self.statistics_.items():
        if col in result.columns:
            result[col] = result[col].fillna(value)
    return result
```

</details>

In [ ]:
# GRADED CLASS: ConfigurableImputer
class ConfigurableImputer:
    """
    A configurable imputer that supports mean, median, mode, and constant strategies.

    After calling fit(X), self.statistics_ is a dict mapping column name to fill value.
    """

    ### START CODE HERE ###

    def __init__(self, strategy='mean', fill_value=None):
        self.strategy   = None  # store strategy
        self.fill_value = None  # store fill_value

    def fit(self, X):
        """
        Compute and store imputation statistics from X.

        Args:
            X (pd.DataFrame or array): Training data.
        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.statistics_ = {}
        for col in df.columns:
            pass  # fill in statistics_[col] based on self.strategy
        return self

    def transform(self, X):
        """
        Apply stored statistics to fill missing values.

        Args:
            X (pd.DataFrame or array): Data to transform.
        Returns:
            pd.DataFrame: Imputed copy of X.
        Raises:
            ValueError: if fit() has not been called.
        """
        if not hasattr(self, 'statistics_'):
            raise ValueError('Imputer has not been fitted yet. Call fit() first.')
        result = pd.DataFrame(X).copy()
        # apply statistics_
        return result

    def fit_transform(self, X):
        """
        Fit on X and immediately transform it.
        """
        return self.fit(X).transform(X)

    ### END CODE HERE ###

In [ ]:
numeric_train = X_train.select_dtypes(include='number')

ci = ConfigurableImputer(strategy='mean')
ci.fit(numeric_train)
print("Learned statistics (mean):")
for col, val in ci.statistics_.items():
    print(f"  {col}: {val:.4f}" if isinstance(val, float) else f"  {col}: {val}")

imputed = ci.transform(numeric_train)
print(f"\nMissing after transform: {imputed.isnull().sum().sum()}")

In [ ]:
# Test your code!
unittests.exercise_1(ConfigurableImputer)

<a name='2---handling-edge-cases'></a>
## 2 - Handling Edge Cases

Production data frequently contains pathological columns: entirely missing columns (imputer fills with 0 or NaN — nonsensical) and constant columns (zero variance — imputation is valid but the column carries no information). Detecting and flagging these columns before modelling prevents silent errors.

<a name='exercise-2---handle_edge_cases'></a>
### **Exercise 2 - `handle_edge_cases`**

**Your Task:**

Implement `handle_edge_cases(col)` that inspects a `pd.Series` and returns a diagnostic dictionary.

**Requirements:**
- Return a `dict` with exactly these keys:
  - `'is_valid'` — `True` if the column can be imputed without concern
  - `'issue_type'` — `None` if valid; `'all_missing'` if every value is NaN; `'constant'` if all non-NaN values are identical

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
if col.isnull().all():
    return {'is_valid': False, 'issue_type': 'all_missing'}
if col.dropna().nunique() == 1:
    return {'is_valid': False, 'issue_type': 'constant'}
return {'is_valid': True, 'issue_type': None}
```

</details>

In [ ]:
# GRADED FUNCTION: handle_edge_cases
def handle_edge_cases(col):
    """
    Detect pathological column patterns that need special treatment.

    Args:
        col (pd.Series): A single column of a DataFrame.

    Returns:
        dict: {'is_valid': bool, 'issue_type': None | 'all_missing' | 'constant'}
    """
    ### START CODE HERE ###

    # Check for all-missing column
    # Check for constant column (all non-null values are the same)
    # Otherwise return valid

    return {'is_valid': None, 'issue_type': None}

    ### END CODE HERE ###

In [ ]:
all_missing_col  = pd.Series([np.nan, np.nan, np.nan])
constant_col     = pd.Series([3.0, 3.0, np.nan, 3.0])
normal_col       = pd.Series([1.0, 2.0, np.nan, 4.0])

print(handle_edge_cases(all_missing_col))   # {'is_valid': False, 'issue_type': 'all_missing'}
print(handle_edge_cases(constant_col))      # {'is_valid': False, 'issue_type': 'constant'}
print(handle_edge_cases(normal_col))        # {'is_valid': True,  'issue_type': None}

In [ ]:
# Test your code!
unittests.exercise_2(handle_edge_cases)

<a name='3---sklearn-pipeline-integration'></a>
## 3 - Sklearn Pipeline Integration

To slot into sklearn `Pipeline` and `ColumnTransformer`, a transformer must inherit from `BaseEstimator` and `TransformerMixin`. Inheriting `TransformerMixin` provides `fit_transform()` for free; inheriting `BaseEstimator` provides `get_params()` / `set_params()`, enabling grid search.

<a name='exercise-3---sklearnimputer'></a>
### **Exercise 3 - `SklearnImputer`**

**Your Task:**

Implement `SklearnImputer(strategy='mean')` that inherits from both `BaseEstimator` and `TransformerMixin`.

**Requirements:**
- `__init__(self, strategy='mean')` — store `strategy` as `self.strategy`
- `fit(self, X, y=None)` — compute per-column fill values, store in `self.fill_values_` dict, return `self`
- `transform(self, X)` — apply `self.fill_values_` and return a `pd.DataFrame`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Key:** When inheriting `BaseEstimator`, do **not** call `super().__init__()` with args. Store all parameters as simple attributes with the same name as the `__init__` parameter (sklearn's `get_params()` relies on this convention).

```python
class SklearnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='mean'):
        self.strategy = strategy  # MUST match __init__ param name!

    def fit(self, X, y=None):
        df = pd.DataFrame(X)
        self.fill_values_ = {}
        for col in df.columns:
            if self.strategy == 'mean':   self.fill_values_[col] = df[col].mean()
            elif self.strategy == 'median': self.fill_values_[col] = df[col].median()
            else: self.fill_values_[col] = df[col].mode()[0]
        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy()
        for col, val in self.fill_values_.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        return df
```

</details>

In [ ]:
# GRADED CLASS: SklearnImputer
class SklearnImputer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible imputer. Inheriting TransformerMixin gives fit_transform() for free.
    Inheriting BaseEstimator gives get_params() / set_params() for grid search.
    """

    ### START CODE HERE ###

    def __init__(self, strategy='mean'):
        self.strategy = None  # store strategy

    def fit(self, X, y=None):
        """
        Compute and store fill values per column.

        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.fill_values_ = {}
        for col in df.columns:
            pass  # compute fill value for each column
        return self

    def transform(self, X):
        """
        Fill missing values using stored fill_values_.

        Returns:
            pd.DataFrame: Imputed copy of X.
        """
        df = pd.DataFrame(X).copy()
        # apply fill_values_
        return df

    ### END CODE HERE ###

In [ ]:
si = SklearnImputer(strategy='median')
X_num_train = X_train.select_dtypes(include='number')
result = si.fit_transform(X_num_train)  # fit_transform is free from TransformerMixin
print(f"Missing after transform: {result.isnull().sum().sum()}")
print(f"get_params():            {si.get_params()}")

In [ ]:
# Test your code!
unittests.exercise_3(SklearnImputer)

<a name='4---fittransform-pattern'></a>
## 4 - Fit/Transform Pattern

The `ProductionImputer` enforces **strict data-leakage prevention** by raising a `RuntimeError` if `transform()` is called before `fit()`. It also tracks whether the imputer has been fitted so that accidental double-fitting (which would silently overwrite learned statistics) can be detected.

<a name='exercise-4---productionimputer'></a>
### **Exercise 4 - `ProductionImputer`**

**Your Task:**

Implement `ProductionImputer(strategy='mean')` that:
- Tracks whether `fit()` has been called via `self.is_fitted_`
- Raises `RuntimeError('Imputer not fitted. Call fit() before transform.')` if `transform()` is called before `fit()`
- `fit()` learns statistics and sets `self.is_fitted_ = True`; returns `self`
- `transform()` applies stored statistics; returns `pd.DataFrame`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
def __init__(self, strategy='mean'):
    self.strategy   = strategy
    self.is_fitted_ = False

def fit(self, X):
    df = pd.DataFrame(X)
    self.statistics_ = {}
    for col in df.columns:
        self.statistics_[col] = df[col].mean()  # simplified
    self.is_fitted_ = True
    return self

def transform(self, X):
    if not self.is_fitted_:
        raise RuntimeError('Imputer not fitted. Call fit() before transform.')
    ...
```

</details>

In [ ]:
# GRADED CLASS: ProductionImputer
class ProductionImputer:
    """
    An imputer that strictly enforces the fit-before-transform pattern.
    Raises RuntimeError if transform() is called before fit().
    """

    ### START CODE HERE ###

    def __init__(self, strategy='mean'):
        self.strategy   = None  # store strategy
        self.is_fitted_ = None  # set to False initially

    def fit(self, X):
        """
        Learn statistics from X. Sets self.is_fitted_ = True.
        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.statistics_ = {}
        for col in df.columns:
            pass  # compute fill value based on self.strategy
        None  # set is_fitted_ = True
        return self

    def transform(self, X):
        """
        Fill missing values. Raises RuntimeError if not fitted.
        Returns:
            pd.DataFrame: Imputed copy of X.
        """
        if not self.is_fitted_:
            raise RuntimeError('Imputer not fitted. Call fit() before transform.')
        df = pd.DataFrame(X).copy()
        # apply statistics_
        return df

    ### END CODE HERE ###

In [ ]:
pi = ProductionImputer(strategy='mean')
print(f"is_fitted_ before fit: {pi.is_fitted_}")

# Should raise RuntimeError
try:
    pi.transform(X_num_train)
    print("ERROR: Should have raised RuntimeError!")
except RuntimeError as e:
    print(f"Correctly raised RuntimeError: {e}")

pi.fit(X_num_train)
print(f"is_fitted_ after fit:  {pi.is_fitted_}")
out = pi.transform(X_num_train)
print(f"Missing after transform: {out.isnull().sum().sum()}")

In [ ]:
# Test your code!
unittests.exercise_4(ProductionImputer)

<a name='5---imputation-logging'></a>
## 5 - Imputation Logging

In production, **every data transformation must be auditable**. The `ImputationLogger` records one entry per imputed column so that data scientists can later inspect which values were substituted, how many cells were affected, and whether the imputation strategy looks reasonable given the data distribution.

<a name='exercise-5---imputationlogger'></a>
### **Exercise 5 - `ImputationLogger`**

**Your Task:**

Implement an `ImputationLogger` class that **both imputes missing values and logs every individual imputation**:
- `__init__(strategy='mean')` — store the strategy; initialise `self.imputation_log_` as an empty `pd.DataFrame`
- `fit(X)` — learn per-column fill statistics into `self.statistics_` (a dict mapping column name → fill value); return `self`
- `transform(X)` — fill missing values **and** log each imputed cell; store the complete log as a `pd.DataFrame` in `self.imputation_log_`; return the imputed `pd.DataFrame`
- `fit_transform(X)` — call `self.fit(X).transform(X)` and return the result

Each row of `self.imputation_log_` must contain exactly these keys:
- `'row'` — the row index of the imputed cell
- `'column'` — the column name
- `'fill_value'` — the value substituted
- `'strategy'` — the imputation strategy used

Supported strategies: `'mean'`, `'median'`, `'mode'`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
def transform(self, X):
    df = pd.DataFrame(X).copy()
    log_entries = []
    for col in df.columns:
        fill_val = self.statistics_[col]
        missing_rows = df.index[df[col].isnull()].tolist()
        for row_idx in missing_rows:
            log_entries.append({'row': row_idx, 'column': col,
                                'fill_value': fill_val, 'strategy': self.strategy})
        df[col] = df[col].fillna(fill_val)
    self.imputation_log_ = pd.DataFrame(log_entries)
    return df
```

</details>

In [ ]:
# GRADED CLASS: ImputationLogger

class ImputationLogger:
    """
    An imputer that logs every individual missing value it fills.

    After fit_transform(X), self.imputation_log_ is a pd.DataFrame with
    one row per imputed cell: {'row', 'column', 'fill_value', 'strategy'}.
    """

    ### START CODE HERE ###

    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.imputation_log_ = None  # will become a pd.DataFrame after transform

    def fit(self, X):
        # Learn per-column fill statistics and store in self.statistics_
        pass

    def transform(self, X):
        # Impute missing values and log each individual imputed cell
        # Store log as a pd.DataFrame in self.imputation_log_
        pass

    def fit_transform(self, X):
        pass  # call fit then transform

    ### END CODE HERE ###

In [ ]:
X_num_with_missing = X_train.select_dtypes(include='number').copy()
logger = ImputationLogger(strategy='mean')
X_imputed = logger.fit_transform(X_num_with_missing)

print(f"Imputed shape:    {X_imputed.shape}")
print(f"Missing after:    {X_imputed.isnull().sum().sum()}")
print(f"\nImputation log ({len(logger.imputation_log_)} entries):")
print(logger.imputation_log_.head(10))

In [ ]:
# Test your code!
unittests.exercise_5(ImputationLogger)

<a name='6---complete-preprocessing-pipeline'></a>
## 6 - Complete Preprocessing Pipeline

A production pipeline chains imputation, scaling, and encoding into a single sklearn `Pipeline` that can be fitted once and serialised. Numeric and categorical columns are handled separately using `ColumnTransformer`.

<a name='exercise-6---create_preprocessing_pipeline'></a>
### **Exercise 6 - `create_preprocessing_pipeline`**

**Your Task:**

Implement `create_preprocessing_pipeline(numeric_cols=None, cat_cols=None)` that returns a sklearn `Pipeline` with **at least 2 steps**:
1. Step `'preprocessor'` — a `ColumnTransformer` that applies:
   - Numeric branch: `SimpleImputer(strategy='mean')` → `StandardScaler()`
   - Categorical branch: `SimpleImputer(strategy='most_frequent')` → `OneHotEncoder(handle_unknown='ignore')`
2. Step `'passthrough'` — a `FunctionTransformer()` identity step

**Requirements:**
- When `numeric_cols=None`, auto-detect numeric columns using `make_column_selector(dtype_include=np.number)`
- When `cat_cols=None`, auto-detect categorical columns using `make_column_selector(dtype_include=object)`
- The function must work when called with **no arguments**: `create_preprocessing_pipeline()`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
if numeric_cols is None:
    numeric_cols = make_column_selector(dtype_include=np.number)
if cat_cols is None:
    cat_cols = make_column_selector(dtype_include=object)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')),
                      ('scaler',  StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('passthrough',  FunctionTransformer())
])
return pipeline
```

</details>

In [ ]:
# GRADED FUNCTION: create_preprocessing_pipeline
def create_preprocessing_pipeline(numeric_cols=None, cat_cols=None):
    """
    Build a complete preprocessing pipeline for mixed-type data.

    Numeric:     SimpleImputer(mean)           → StandardScaler
    Categorical: SimpleImputer(most_frequent)  → OneHotEncoder

    Args:
        numeric_cols (list | None): Column names of numeric features.
                                    Auto-detected when None.
        cat_cols     (list | None): Column names of categorical features.
                                    Auto-detected when None.

    Returns:
        sklearn.pipeline.Pipeline: A 2-step pipeline (preprocessor + passthrough).
    """
    ### START CODE HERE ###

    # Auto-detect column types when not provided
    # Build numeric transformer: impute → scale
    # Build categorical transformer: impute → encode
    # Combine with ColumnTransformer
    # Return 2-step Pipeline (preprocessor + FunctionTransformer passthrough)

    return None

    ### END CODE HERE ###

In [ ]:
# Call with no args — column types are auto-detected from the training data
pipeline = create_preprocessing_pipeline()
pipeline.fit(X_train)

X_tr_processed = pipeline.transform(X_train)
X_te_processed = pipeline.transform(X_test)
print(f"Processed training shape: {X_tr_processed.shape}")
print(f"Pipeline steps:           {pipeline.steps}")

In [ ]:
# Test your code!
unittests.exercise_6(create_preprocessing_pipeline)

## Serialisation with Joblib

Once the pipeline is fitted, save it with `joblib.dump()` to persist the learned statistics. Load with `joblib.load()` to reproduce identical transformations at serving time.

In [ ]:
# Save and reload the fitted pipeline
joblib.dump(pipeline, '/tmp/imputation_pipeline.joblib')
loaded_pipeline = joblib.load('/tmp/imputation_pipeline.joblib')

X_te_loaded = loaded_pipeline.transform(X_test)
print(f"Loaded pipeline output shape: {X_te_loaded.shape}")
print("Outputs match:", np.allclose(X_tr_processed, pipeline.transform(X_train)))